In [ ]:
# --- content before the first `# In[]:` marker (e.g. notebook title / markdown) ---
# # MNIST digits recognition with multi class SVM 

In [1]:
# In[1]:


import pandas as pd
import sklearn as sk
from sklearn import svm, decomposition, preprocessing
import numpy as np


# ### Load the training data

In [2]:
# In[2]:


train = pd.read_csv("../input/train.csv")
#train = train.as_matrix()
X = train.values[:,1:]
Y = train.ix[:,0]
print((len(Y)))


# The first column of the training data contains labels for each image

In [3]:
# In[3]:


import matplotlib.pyplot as plt
from skimage.transform import rotate
from sklearn.cross_validation import train_test_split

# Split into train / validation BEFORE any augmentation or preprocessing, so that
# rotated copies of validation images never end up in the training set and the
# scaler/PCA statistics are learned from training data only (no leakage).
X_train, X_test, Y_train, Y_test = train_test_split(X, np.asarray(Y, dtype=float), random_state = 7,
                                                    test_size = 0.3)

n = len(Y_train)
print(n)

# Augment only the training split with random rotations
random_indexes = np.random.randint(0, high = n, size = 10000 )
X_train = np.vstack((X_train, np.zeros((10000, X_train.shape[1]))))
Y_train = np.concatenate([Y_train,np.zeros((10000))])
print((Y_train.shape))
for ind in random_indexes:  
    img = X_train[ind, :].reshape(28,28)
    angle = np.random.randint(-30, 30)
    img_rot = rotate(img, angle)
    X_train[n,:] = img_rot.reshape(1,784)
    Y_train[n] = Y_train[ind]
    n = n + 1
print(("size of training set after random rotations: ", X_train.shape))
print((len(Y_train)))
plt.title("last of added training examples, label: " + str(Y_train[-1]))
img1 = plt.imshow(X_train[n-1,:].reshape(28,28), cmap=plt.cm.gray_r, interpolation="nearest")


# Fit the scaler on the training split only, then apply it to the validation split
scaler = preprocessing.StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# ### Reduce dimensionality

# Perform PCA to reduce size of feature vectors


In [4]:
# In[4]:


COMPONENTS_NUM = 300

# Fit PCA on the training split only
pca = decomposition.PCA(n_components = 0.8, whiten = True)
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)


# ### Train SVM multiclass classificator


In [5]:
# In[5]:


classifier = svm.SVC()
classifier.fit(X_train, Y_train)
score = classifier.score(X_test, Y_test)
print(('Score \n', score))


# ### Load test data and predict labels


In [6]:
# In[6]:


test  = pd.read_csv("../input/test.csv")
test = test.values
# Use the scaler fitted on the training data (not statistics of the test set itself)
test = scaler.transform(test)
#pca = decomposition.PCA(components_num)
X_test_red = pca.transform(test)
Y_pred = classifier.predict(X_test_red)


# ### Save predictions to predict.csv file


In [7]:
# In[7]:


pd.DataFrame({"ImageId": list(range(1,len(Y_pred)+1)), "Label": Y_pred}).to_csv('out.csv', index=False, header=True)    